# R15-H157 - Corpus-class transfer: the single-corpus overfit test

**Author**: Claude (opus executor)  &nbsp;|&nbsp; **Date**: 2026-07-08

**Hypothesis** - on a structurally different technical corpus (scientific-paper reference library,
30 papers on ML / IR / KG methods) the engine's *lifecycle machinery transfers* (curing gate fires
and holds, FSM reaches STABLE, no spurious recures) but the *identity calibration does NOT transfer*
within tolerance (ECE degrades > 2x its benchmark-corpus reference) - i.e. the shipped isotonic curve
is dataset-bound and calibration must be re-estimated per corpus.

**Analysis** - three registered clauses, all read-only against the H157 graph on the default instance:

1. **Blind pair labeling (H101)** - sample 60 resolution decisions (20 merge / 20 block / 20 defer),
   label SAME / DIFFERENT / UNSURE from entity evidence *blind to the resolver's decision*, then
   unblind and compute precision / recall proxies
2. **ECE transfer** - bin the resolver's logged posteriors (10 bins) against the blind labels and
   compare calibration error to the shipped artifact's benchmark reference ECE
3. **Lifecycle** - reconstruct the FSM / curing lifecycle from the event stream against the registered
   lifecycle clause

Constraint: the graph is treated STRICTLY READ-ONLY (MATCH only); the vLLM endpoint is never called.

## Imports

In [1]:
from __future__ import annotations
import os, json, math, random
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from neo4j import GraphDatabase
from rich import print as rp
from rich.table import Table

PROJ = Path("..").resolve()
load_dotenv(PROJ / ".env")

True

## Configuration

Verify the DEF-4/DEF-5 pinning: the H157 config pins the Neo4j URI and it must resolve to the same
instance the ingest used (the default `.env` instance). The reference ECE is the shipped calibration's
held-out calibration error on the benchmark (device) corpus, recorded by the R12-H129 transfer report.

In [2]:
CONFIG_PATH   = PROJ / "config-h157-papers.yml"
ARTIFACT_PATH = PROJ / "data/processed/identity-calibration-v2.json"
EVENTS_PATH   = PROJ / "logs/h157-events.jsonl"
H129_REPORT   = PROJ / "reports/calibration-h129-transfer-20260707-100316.json"
LOG_PATH      = PROJ / "logs/h157-analysis.log"

cfg = yaml.safe_load(CONFIG_PATH.read_text())
CFG_URI = cfg["neo4j"]["uri"]
ENV_URI = os.environ["NEO4J_URI"]
assert CFG_URI == ENV_URI, f"config URI {CFG_URI} != env URI {ENV_URI} - would touch a different instance"

ref = json.loads(H129_REPORT.read_text())
REFERENCE_ECE = ref["ece"]              # shipped isotonic held-out ECE on the benchmark corpus
BENCHMARK_TYPES = 12                    # cured ontology size on the device corpus (v28)

artifact = json.loads(ARTIFACT_PATH.read_text())
rp(f"[bold]Neo4j URI pinned & verified[/bold]: {CFG_URI}")
rp(f"calibration artifact {artifact['version']}  (isotonic in-sample ECE={artifact['provenance']['isotonic_ece']}, "
   f"held-out ref ECE={REFERENCE_ECE:.4f} @ n={ref['n']})")

# id-pair-keyed blind labels, frozen BEFORE unblinding (S=same, D=different, U=unsure)
FROZEN_LABELS = json.loads(r'''{"e_219a82edf2012c28::e_6d49ff190bc00332":"D","e_df8400013ae9fde3::e_1f241102c9876846":"D","e_468da084e9953050::e_1cd44f557040b53f":"S","e_891e3cfc3bb54178::e_a4d86bbbc7ca2a06":"D","e_a6f9405c3e1c58c5::e_e32d82c5d09592ae":"D","e_ba558b36f81517e8::e_c1a84f0eb369fd2d":"S","e_faacd5a9d2304fba::e_8aed40c92b7ddf90":"S","e_7b77537bc78547cf::e_1569785aa7fea188":"U","e_306830a4269cd028::e_05399bf406d75743":"D","e_93a63bdead1d1ead::e_8081bb323c8e8aee":"S","e_90eee9c0776a24d9::e_a50601a8243a242a":"D","e_612050f4f3e46bd8::e_bae34d00c19eaeaf":"D","e_e752bc94938bc0ca::e_5444223ac799ba8a":"D","e_b3e1a95b065da49e::e_9ba0073e6812be44":"S","e_4cddc093d54bb1a4::e_2a6bf1ec783ebb57":"S","e_150a8af76a92892f::e_de4bedce64a13c43":"D","e_54bf6447f6a0b4a7::e_3a6ba7ad8fbc2fc6":"S","e_0feb88eee2c78b40::e_eb89a7d82b69cedf":"S","e_4d816b7c7f47ef40::e_e4528017d808495f":"S","e_ebb05f32f532fe3b::e_049efa8c67d76a17":"D","e_4cddc093d54bb1a4::e_575964e711b00bb1":"S","e_4669d443b3d9b2d9::e_d8dfa8ce0e2ad58a":"D","e_5d04917d02569490::e_7b9b03d8db20bd4b":"D","e_c5375c2d1d8a235e::e_48a037298d6bace3":"D","e_99fc756bb5b67ae2::e_7baa3572bb168282":"S","e_a36e3d3c4e20d6e0::e_1aa15721b46b8a29":"D","e_021b56b37ecba481::e_6c15311905b5939b":"S","e_908019742833a0fc::e_7ba8569be15c20e4":"D","e_ccc40eba1e538f90::e_085587f747f13c2d":"D","e_b5018133b294c6d3::e_6ed13dd00334ae46":"D","e_205f7a0c8ee884f3::e_c6ce69c44e3fcc7b":"S","e_ead422c6ef1f86a4::e_d55c419cd784e90e":"U","e_31cf0606b155cfba::e_29aab777ad1fa518":"S","e_f93b510985410767::e_24dd7672b1172d35":"S","e_aa81106aa78b9af1::e_2eb5b1c7bcf97d48":"S","e_b71527dbed928130::e_70bc0723a9bdf3ef":"D","e_ba376a42baffd4d5::e_ccdd6bb481c97816":"S","e_e6d245e3e64d74a2::e_697990aece2eb22a":"D","e_6921f55281de85a3::e_086bbb098446452a":"D","e_b181e160dad5bca5::e_bcd3d415b0f6f78d":"S","e_60e5213ad2f7a4bb::e_81f16a91d3d387f7":"D","e_3535d0d6c13e4020::e_a3d6f8191bdee8a1":"S","e_f17ca7b1d7cd8924::e_8c8db1952f687e65":"S","e_cf3eab699c5a417f::e_8b658d6963ee98a9":"S","e_fd18714637a24918::e_001bc1b7617d926c":"S","e_14455a2796bab413::e_e4b471fa5e158acd":"U","e_49917ae4c100b5d2::e_3cf10965c09ab94e":"U","e_a63b60eea4665f00::e_6b4e2582d4726d49":"D","e_69a1a7fe923c0a50::e_329502c486359c19":"D","e_a131a8ccfcebe81f::e_47fc4a61117fae4a":"S","e_b3785ae0cb10ea47::e_ec42333dd5967dfb":"D","e_fd7b2448eec76a0c::e_81f16a91d3d387f7":"D","e_f7115f42ebbb13b3::e_3b8004aff0ff86ee":"D","e_91f4f7cc554296bd::e_eabb1772d87120c7":"S","e_a1e779dbf3ca872c::e_9286fc8bb007ce24":"D","e_f7f72c0bda754eb1::e_5126106a21f8ebff":"S","e_2c2105e436f8b945::e_c1188c5d261b6ff0":"S","e_17280886d5567f58::e_20f6b3db3aec1734":"D","e_4fc421d7da53a52d::e_4e7b08952acae50a":"D"}''')
rp(f"frozen blind labels: {len(FROZEN_LABELS)} pairs")

Neo4j URI pinned & verified: bolt://user-konrad.jelen-kgf-neo4j:7687

calibration artifact v2  (isotonic in-sample ECE=0.0, held-out ref ECE=0.0503 @ n=297)

frozen blind labels: 59 pairs

## Event stream census - headline transfer signals

The registration expects the transfer corpus to look structurally different from the benchmark:
more entity types (academic genre yields methods / datasets / metrics / authors), a heavier
block-dominated resolution mix, and extraction warnings from the paper-genre text.

In [3]:
def load_events(path):
    out = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line:
            out.append(json.loads(line))
    return out

events = load_events(EVENTS_PATH)
ev_counts = Counter(e["event"] for e in events)

decision_mix = {d: ev_counts[f"resolution.{d}"] for d in ("merge", "block", "defer")}
n_veto      = ev_counts["resolution.veto"]
n_warn      = ev_counts["extraction.warning"]
n_emerged   = ev_counts["ontology.type_emerged"]
n_confirmed = ev_counts["ontology.type_confirmed"]

warn_kinds = Counter()
for e in events:
    if e["event"] == "extraction.warning":
        r = e.get("reason", "")
        if r.startswith("dangling relationship"):      warn_kinds["dangling_relationship"] += 1
        elif r.startswith("self-referencing"):          warn_kinds["self_referencing"] += 1
        elif "Mode" in r and "not registered" in r:     warn_kinds["chunk_extract_mode_error"] += 1
        elif "litellm" in r or "litel" in r:            warn_kinds["chunk_extract_litellm"] += 1
        else:                                           warn_kinds["other"] += 1

rp(f"resolution mix  merge/block/defer = {decision_mix['merge']} / {decision_mix['block']} / {decision_mix['defer']}")
rp(f"resolution.veto (NLI) = {n_veto}   extraction.warning = {n_warn}")
rp(f"warning breakdown: {dict(warn_kinds)}")
rp(f"ontology: type_emerged={n_emerged}  type_confirmed={n_confirmed}")

resolution mix  merge/block/defer = 2050 / 14270 / 1689

resolution.veto (NLI) = 24   extraction.warning = 1085

warning breakdown: {'chunk_extract_mode_error': 20, 'dangling_relationship': 1043, 'self_referencing': 17, 
'chunk_extract_litellm': 5}

ontology: type_emerged=66  type_confirmed=55

## Clause 3 - Lifecycle transfer

Reconstruct the FSM path and the curing trajectory. The registered clause passes when the curing gate
fires and holds, no spurious recures occur (no STABLE -> CURING), and the FSM reaches STABLE.

In [4]:
fsm = [(e["ts"], e["state"]) for e in events if e["event"] == "fsm.transition"]
cured = [e for e in events if e["event"] == "curing.cured"]
metrics = [e for e in events if e["event"] == "curing.metrics"]

states = [s for _, s in fsm]
reached_stable = "STABLE" in states
recure = any(states[i] == "CURING" and "STABLE" in states[:i] for i in range(len(states)))
cure_point = cured[0]["documents"] if cured else None
cured_types = cured[0]["types"] if cured else None
cured_entities = cured[0]["entities"] if cured else None

traj = [(m["doc_index"], m["unique_types"]) for m in metrics]
final_types = int(traj[-1][1]) if traj else cured_types

rp(f"[bold]FSM path[/bold]: {' -> '.join(['EMPTY'] + states)}")
rp(f"reached STABLE: {reached_stable}   spurious recure (STABLE->CURING): {recure}")
rp(f"curing.cured: reason={cured[0]['reason']!r}  cure_point=doc {cure_point}  "
   f"types={cured_types}  entities={cured_entities}")
rp(f"type-count trajectory: first={traj[0] if traj else None}  cured={cure_point}:{cured_types}  "
   f"final~{final_types}  (benchmark cured ~doc 4, {BENCHMARK_TYPES} types)")

lifecycle = {
    "fsm_path": ["EMPTY"] + states,
    "reached_stable": bool(reached_stable),
    "spurious_recure": bool(recure),
    "cure_point_doc": cure_point,
    "cured_types": cured_types,
    "cured_entities": cured_entities,
    "final_types": int(final_types) if final_types else None,
    "n_curing_metrics": len(metrics),
}
clause3_pass = reached_stable and not recure and cure_point is not None
rp(f"[bold]Clause 3 (lifecycle) PASS = {clause3_pass}[/bold]  (STABLE reached, gate held, no recure)")

FSM path: EMPTY -> INITIALIZING -> CURING -> CURING -> STABLE

reached STABLE: True   spurious recure (STABLE->CURING): False

curing.cured: reason='converged'  cure_point=doc 13  types=51  entities=5863

type-count trajectory: first=(0, 15.0)  cured=13:51  final~51  (benchmark cured ~doc 4, 12 types)

Clause 3 (lifecycle) PASS = True  (STABLE reached, gate held, no recure)

## Clause 1 - Blind pair labeling (H101 protocol)

Sample 60 logged resolution decisions, stratified 20 merge / 20 block / 20 defer. Merged mentions that
were collapsed into a canonical node lose their id; to keep the evidence honest the sample is drawn only
from decisions whose *both* endpoints still resolve to a name (via `Entity` or `KGFEntityVersion`). Only
~17% of merges are fully recoverable, so the merge stratum is a documented subset - a limitation noted in
the verdict. Blind labels were frozen (above) from names + descriptions alone, decision hidden.

In [5]:
driver = GraphDatabase.driver(ENV_URI, auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))

lut = {}
with driver.session() as s:
    for lab_ in ("Entity", "KGFEntityVersion"):
        for x in s.run(f"MATCH (e:{lab_}) RETURN e.id AS id, e.name AS name, e.description AS d, "
                       f"e.source_documents AS sd"):
            lut.setdefault(x["id"], {"name": x["name"], "desc": (x["d"] or "")[:200], "sd": x["sd"]})
rp(f"evidence LUT: {len(lut)} resolvable ids")

evidence LUT: 10100 resolvable ids

In [6]:
byd = {"merge": [], "block": [], "defer": []}
for e in events:
    if e["event"] in ("resolution.merge", "resolution.block", "resolution.defer"):
        if e["left_id"] in lut and e["right_id"] in lut:
            byd[e["decision"]].append(e)
rp({k: len(v) for k, v in byd.items()})

random.seed(157)
sample = []
for d, n in (("merge", 20), ("block", 20), ("defer", 20)):
    sample += random.sample(byd[d], n)
random.seed(999)
random.shuffle(sample)

rows = []
for p in sample:
    L, R = lut[p["left_id"]], lut[p["right_id"]]
    key = f"{p['left_id']}::{p['right_id']}"
    rows.append({
        "key": key, "decision": p["decision"], "posterior": round(p["posterior"], 4),
        "lname": L["name"], "rname": R["name"], "ldesc": L["desc"], "rdesc": R["desc"],
        "label": FROZEN_LABELS[key],
    })
pairs = pd.DataFrame(rows)
assert set(pairs["key"]) <= set(FROZEN_LABELS), "sample drifted from frozen labels"
assert len(pairs) == 60
rp(f"sampled & labeled {len(pairs)} pairs; strata = {dict(Counter(pairs['decision']))}")

{'merge': 352, 'block': 13939, 'defer': 1593}

sampled & labeled 60 pairs; strata = {'block': 20, 'defer': 20, 'merge': 20}

### Unblinding - precision / recall proxies

Map the resolver's action to a prediction: **merge -> SAME**, **block -> DIFFERENT**, **defer -> abstain**.
Score against the blind labels, dropping UNSURE from the decisive rates. Because the strata are balanced
by construction (not natural prevalence) these are *proxies*, not population rates.

In [7]:
strat = defaultdict(Counter)
for _, r in pairs.iterrows():
    strat[r["decision"]][r["label"]] += 1

show = Table(title="Blind-label distribution per resolver decision")
show.add_column("resolver decision"); show.add_column("SAME", justify="right")
show.add_column("DIFFERENT", justify="right"); show.add_column("UNSURE", justify="right")
for d in ("merge", "block", "defer"):
    show.add_row(d, str(strat[d]["S"]), str(strat[d]["D"]), str(strat[d]["U"]))
rp(show)

mS, mD = strat["merge"]["S"], strat["merge"]["D"]
bS, bD = strat["block"]["S"], strat["block"]["D"]
merge_precision = mS / (mS + mD) if (mS + mD) else float("nan")
block_correct   = bD / (bS + bD) if (bS + bD) else float("nan")
recall_same     = mS / (mS + bS) if (mS + bS) else float("nan")
defer_ambiguity = (strat["defer"]["U"] + min(strat["defer"]["S"], strat["defer"]["D"])) / \
                  sum(strat["defer"].values())

false_merges = [(r["lname"], r["rname"], r["posterior"]) for _, r in pairs.iterrows()
                if r["decision"] == "merge" and r["label"] == "D"]
false_blocks = [(r["lname"], r["rname"]) for _, r in pairs.iterrows()
                if r["decision"] == "block" and r["label"] == "S"]

rp(f"[bold]merge precision proxy[/bold] = {merge_precision:.3f}  ({mS}/{mS+mD} merges truly SAME)")
rp(f"[bold]block correctness[/bold]     = {block_correct:.3f}  ({bD}/{bS+bD} blocks truly DIFFERENT)")
rp(f"[bold]recall proxy (SAME)[/bold]   = {recall_same:.3f}")
rp(f"defer ambiguity (near-50/50 + unsure) = {defer_ambiguity:.3f}")
for ln, rn, po in false_merges:
    rp(f"  false MERGE: {ln!r} || {rn!r}  posterior={po}")
clause1_pass = (merge_precision >= 0.7 and block_correct >= 0.7)
rp(f"[bold]Clause 1 (resolver decision quality) PASS = {clause1_pass}[/bold]")

 Blind-label distribution per resolver decision  
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ resolver decision ┃ SAME ┃ DIFFERENT ┃ UNSURE ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ merge             │   19 │         1 │      0 │
│ block             │    0 │        19 │      1 │
│ defer             │    8 │         9 │      3 │
└───────────────────┴──────┴───────────┴────────┘

merge precision proxy = 0.950  (19/20 merges truly SAME)

block correctness     = 1.000  (19/19 blocks truly DIFFERENT)

recall proxy (SAME)   = 1.000

defer ambiguity (near-50/50 + unsure) = 0.550

false MERGE: 'Yu Zhang' || 'Yue Zhang'  posterior=0.2768

Clause 1 (resolver decision quality) PASS = True

## Clause 2 - ECE transfer

The registered clause: bin the resolver's logged posteriors (10 equal-width bins), take the gap between
mean posterior and empirical SAME-rate per bin (weighted), and compare to the benchmark reference ECE.
**Branch**: transfer ECE > 2 x reference -> shipped calibration does NOT transfer, must be re-estimated
per corpus. UNSURE pairs are excluded (no ground truth). The logged posterior is the resolver's operational
Bayesian posterior - the probability it actually acts on - so this measures operational calibration on the
new corpus.

In [8]:
def ece_10bin(probs, y, bins=10):
    probs = np.asarray(probs, float); y = np.asarray(y, float)
    N = len(y); e = 0.0; detail = []
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        m = (probs >= lo) & (probs < hi if b < bins - 1 else probs <= hi)
        if m.sum() == 0:
            continue
        mp, fy = probs[m].mean(), y[m].mean()
        e += m.sum() / N * abs(mp - fy)
        detail.append((round(lo, 1), int(m.sum()), round(float(mp), 3), round(float(fy), 3)))
    return e, detail

lab = pairs[pairs["label"] != "U"].copy()
lab["y"] = (lab["label"] == "S").astype(int)
transfer_ece, detail = ece_10bin(lab["posterior"], lab["y"])
ratio = transfer_ece / REFERENCE_ECE
branch = "BREAK: calibration must be re-estimated per corpus" if ratio > 2 else \
         "HOLD: shipped constants suffice for this corpus class"

dt = Table(title=f"ECE reliability (10 bins, n={len(lab)} labeled)")
dt.add_column("bin_lo"); dt.add_column("n", justify="right")
dt.add_column("mean posterior", justify="right"); dt.add_column("frac SAME", justify="right")
for lo, n, mp, fy in detail:
    dt.add_row(str(lo), str(n), str(mp), str(fy))
rp(dt)
rp(f"[bold]transfer ECE[/bold] = {transfer_ece:.4f}   reference ECE = {REFERENCE_ECE:.4f}   "
   f"ratio = {ratio:.2f}x   (2x threshold)")
rp(f"[bold]branch fired[/bold]: {branch}")
under = (lab["posterior"] - lab["y"]).mean() < 0
clause2_break = ratio > 2
rp(f"direction: posteriors are systematically {'under' if under else 'over'}-confident on the paper corpus")

  ECE reliability (10 bins, n=56 labeled)   
┏━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ bin_lo ┃  n ┃ mean posterior ┃ frac SAME ┃
┡━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ 0.0    │ 17 │          0.031 │       0.0 │
│ 0.1    │ 12 │          0.155 │      0.25 │
│ 0.2    │ 10 │          0.258 │       0.7 │
│ 0.3    │  4 │          0.338 │       1.0 │
│ 0.4    │  5 │          0.448 │       1.0 │
│ 0.5    │  4 │          0.568 │       1.0 │
│ 0.6    │  3 │          0.645 │       1.0 │
│ 0.7    │  1 │          0.702 │       1.0 │
└────────┴────┴────────────────┴───────────┘

transfer ECE = 0.2606   reference ECE = 0.0503   ratio = 5.18x   (2x threshold)

branch fired: BREAK: calibration must be re-estimated per corpus

direction: posteriors are systematically under-confident on the paper corpus

## Verdict and report

The hypothesis is CONFIRMED when lifecycle transfers (clause 3 pass) AND calibration breaks (clause 2
ratio > 2x). Clause 1 characterizes the resolver's decision quality on the new domain - decisions can
remain accurate even while the posterior *probabilities* miscalibrate (a fixed threshold still ranks
correctly), which is itself the argument for a self-calibration path rather than shipped constants.

In [9]:
driver.close()

lifecycle_transfers = clause3_pass
calibration_breaks  = clause2_break
hypothesis_confirmed = lifecycle_transfers and calibration_breaks

if hypothesis_confirmed:
    verdict = "CONFIRMED"
    rec = ("Lifecycle machinery transfers to the paper corpus (FSM reached STABLE, curing gate held at "
           f"doc {lifecycle['cure_point_doc']}, no recure) but the shipped isotonic calibration does NOT "
           f"transfer (ECE {transfer_ece:.3f} = {ratio:.1f}x the {REFERENCE_ECE:.3f} benchmark reference, "
           "well past 2x). Ship a per-corpus self-calibration path (H142 lineage); do not reuse the device-"
           "corpus isotonic constants across corpus classes. Resolver DECISIONS still transfer well "
           f"(merge precision {merge_precision:.2f}, block {block_correct:.2f}); the failure is in the "
           "posterior probability semantics, and the residual false-merge surface is author-name morphology.")
elif lifecycle_transfers and not calibration_breaks:
    verdict = "REFUTED (pleasantly)"
    rec = "Calibration held within 2x - shipped constants suffice for the corpus classes tested."
else:
    verdict = "MIXED / see clauses"
    rec = "Lifecycle did not cleanly transfer; inspect FSM path before drawing the calibration conclusion."

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "hypothesis": "R15-H157",
    "phase": "ANALYSIS",
    "generated_utc": stamp,
    "author": "Claude (opus executor)",
    "graph_uri": ENV_URI,
    "read_only": True,
    "headline": {
        "documents_ingested": 30,
        "documents_loaded": 26,
        "entities_extracted": 16547,
        "relationships": 12203,
        "cured_types": lifecycle["cured_types"],
        "benchmark_types": BENCHMARK_TYPES,
        "resolution_mix": decision_mix,
        "resolution_veto": n_veto,
        "extraction_warnings": n_warn,
        "warning_breakdown": dict(warn_kinds),
        "type_emerged": n_emerged,
        "type_confirmed": n_confirmed,
    },
    "clause1_blind_labeling": {
        "n_pairs": 60, "strata": {d: dict(strat[d]) for d in ("merge", "block", "defer")},
        "merge_precision_proxy": round(merge_precision, 4),
        "block_correctness": round(block_correct, 4),
        "recall_same_proxy": round(recall_same, 4),
        "defer_ambiguity": round(defer_ambiguity, 4),
        "false_merges": false_merges, "false_blocks": false_blocks,
        "merge_stratum_recoverable_only": True,
        "recoverable_merge_fraction": round(len(byd["merge"]) / max(1, sum(1 for e in events
            if e["event"] == "resolution.merge")), 4),
        "pass": bool(clause1_pass),
    },
    "clause2_ece_transfer": {
        "transfer_ece": round(float(transfer_ece), 4),
        "reference_ece": REFERENCE_ECE,
        "ratio": round(float(ratio), 3),
        "threshold": 2.0,
        "n_labeled": int(len(lab)),
        "bins": detail,
        "branch": "break" if clause2_break else "hold",
        "calibration_breaks": bool(calibration_breaks),
    },
    "clause3_lifecycle": lifecycle,
    "clause3_pass": bool(clause3_pass),
    "verdict_recommendation": {"verdict": verdict, "recommendation": rec},
}

out_path = PROJ / f"reports/corpus-transfer-h157-{stamp}.json"
out_path.write_text(json.dumps(report, indent=2))

with open(LOG_PATH, "a") as f:
    f.write(f"[{stamp}] R15-H157 ANALYSIS verdict={verdict} "
            f"lifecycle_pass={clause3_pass} ece={transfer_ece:.4f} ref={REFERENCE_ECE:.4f} "
            f"ratio={ratio:.2f}x branch={report['clause2_ece_transfer']['branch']} "
            f"merge_prec={merge_precision:.3f} block_corr={block_correct:.3f} report={out_path.name}\n")

rp(f"[bold green]VERDICT: {verdict}[/bold green]")
rp(f"report -> {out_path}")
rp(rec)

VERDICT: CONFIRMED

report -> 
/home/lab/workspace/learning/projects/knowledge-graph-foundry/reports/corpus-transfer-h157-20260708T081013Z.json

Lifecycle machinery transfers to the paper corpus (FSM reached STABLE, curing gate held at doc 13, no recure) but 
the shipped isotonic calibration does NOT transfer (ECE 0.261 = 5.2x the 0.050 benchmark reference, well past 2x). 
Ship a per-corpus self-calibration path (H142 lineage); do not reuse the device-corpus isotonic constants across 
corpus classes. Resolver DECISIONS still transfer well (merge precision 0.95, block 1.00); the failure is in the 
posterior probability semantics, and the residual false-merge surface is author-name morphology.